In [1]:
import pandas as pd

df = pd.read_csv("2_preprocessed_sequences_with_interpolation_v0.csv")
df["new_label"]= df["new_label"].fillna(0)


In [2]:
from utils_dataset import create_classical_dataset
df_train_val, df_test = create_classical_dataset(df, test_size =1000, total_train_val_size=200)

In [3]:
df_train_val.head()

,Unnamed: 0,Extracted_Datetime,Dataset_prefix_group_id,Rel_Image_Path,Rel_Label_Path,Image_basename,Label_basename,Origin_dataset_name,Datetime_Str,Extension,...,img_width,has_label,yolo_bbox_xcenter,yolo_bbox_ycenter,yolo_bbox_width,yolo_bbox_height,new_label,pred,nb_detections,group
485,716,2017-07-08 16:52:13,DS_fp_pyronear_brison_1_group_177_1,pyronear_ds_03_2024/images/train/awf_nvseismol...,pyronear_ds_03_2024/labels/train/awf_nvseismol...,awf_nvseismolab_baldcaX-0119_2017_07_08T16_52_...,awf_nvseismolab_baldcaX-0119_2017_07_08T16_52_...,df_pyronear_ds_03_2024_train,2017_07_08T16_52_13,jpg,...,1280,True,0.654688,0.465972,0.207812,0.384722,1.0,NaN,1.0,1
486,717,2017-07-08 16:52:39,DS_fp_pyronear_brison_1_group_177_1,pyronear_ds_03_2024/images/train/awf_nvseismol...,pyronear_ds_03_2024/labels/train/awf_nvseismol...,awf_nvseismolab_baldcaX-0119_2017_07_08T16_52_...,awf_nvseismolab_baldcaX-0119_2017_07_08T16_52_...,df_pyronear_ds_03_2024_train,2017_07_08T16_52_39,jpg,...,1280,True,0.641406,0.459722,0.196875,0.400000,1.0,NaN,1.0,1
487,718,2017-07-08 16:53:10,DS_fp_pyronear_brison_1_group_177_1,pyronear_ds_03_2024/images/train/awf_nvseismol...,pyronear_ds_03_2024/labels/train/awf_nvseismol...,awf_nvseismolab_baldcaX-0119_2017_07_08T16_53_...,awf_nvseismolab_baldcaX-0119_2017_07_08T16_53_...,df_pyronear_ds_03_2024_train,2017_07_08T16_53_10,jpg,...,1280,True,0.628906,0.489583,0.225000,0.343056,1.0,NaN,1.0,1
488,82721,2017-07-08 16:53:38,DS_fp_pyronear_brison_1_group_177_1,pyronear_ds_03_2024/images/train/awf_nvseismol...,pyronear_ds_03_2024/labels/train/awf_nvseismol...,awf_nvseismolab_baldcaX-0119_2017_07_08T16_53_...,awf_nvseismolab_baldcaX-0119_2017_07_08T16_53_...,df_pyronear_ds_03_2024_train,2017_07_08T16_53_38,jpg,...,1280,True,0.622656,0.477083,0.201562,0.368056,1.0,NaN,1.0,1
489,719,2017-07-08 16:53:38,DS_fp_pyronear_brison_1_group_177_1,pyronear_ds_03_2024/images/train/awf_nvseismol...,pyronear_ds_03_2024/labels/train/awf_nvseismol...,awf_nvseismolab_baldcaX-0119_2017_07_08T16_53_...,awf_nvseismolab_baldcaX-0119_2017_07_08T16_53_...,df_pyronear_ds_03_2024_train,2017_07_08T16_53_38,jpg,...,1280,True,0.622656,0.477083,0.201562,0.368056,1.0,NaN,1.0,1


In [4]:
from utils_dataset import prepare_sequences

X_train_test, y_train_test = prepare_sequences(df_train_val)

In [5]:
y_train_test

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_train_test, y_train_test, test_size=0.2, random_state=123)

In [7]:
from utils_model import build_teacher_model

teacher_model = build_teacher_model()

In [8]:
from utils_model import compile_with_loss

compile_with_loss(teacher_model, 1.0)

In [9]:
from utils_dataset import create_tensorflow_dataset

train_dataset, test_dataset = create_tensorflow_dataset(X_train, y_train, X_test, y_test, 1, 100)

In [10]:
from utils_model import MetricsCallback

metrics_callback = MetricsCallback(train_dataset, test_dataset, batch_interval=5)

In [11]:
teacher_history = teacher_model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=3, #3 #20
    callbacks =[metrics_callback]
)

Epoch 1/3


c:\Users\treym\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\backend.py:5805: UserWarning: "`binary_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Sigmoid activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


1/1 [==============================] - 0s 242ms/step


c:\Users\treym\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


1/1 [==============================] - 0s 190ms/step
Batch 0: train_loss=0.3977823628229089, train_accuracy=0.5625, train_precision=0.31640625, train_recall=0.5625, val_loss=0.3165861563757062, val_accuracy=0.25, val_precision=0.0625, val_recall=0.25
 1/32 [..............................] - ETA: 16:03 - loss: 0.1283 - accuracy: 1.0000

...

1/1 [==============================] - 0s 162ms/step
Batch 25: train_loss=0.41993166448082775, train_accuracy=0.71875, train_precision=0.8288043478260869, train_recall=0.71875, val_loss=0.4341838448308408, val_accuracy=0.75, val_precision=0.5625, val_recall=0.75
1/1 [==============================] - 0s 150ms/step
Batch 30: train_loss=0.4594957180088386, train_accuracy=0.8125, train_precision=0.8382936507936508, train_recall=0.8125, val_loss=0.4978315974585712, val_accuracy=0.875, val_precision=0.8928571428571428, val_recall=0.875
1/1 [==============================] - 0s 157ms/step
Epoch 0: train_loss=0.4734990211436525, train_accuracy=0.84375, train_precision=0.8463235294117648, train_recall=0.84375, val_loss=0.49748373683542013, val_accuracy=0.875, val_precision=0.8928571428571428, val_recall=0.875
32/32 [==============================] - 154s 4s/step - loss: 0.0558 - accuracy: 1.0000 - val_loss: 0.1200 - val_accuracy: 0.8750
Epoch 2/3
1/1 [==============================] - 0s 191ms

c:\Users\treym\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


1/1 [==============================] - 0s 177ms/step
Batch 25: train_loss=0.41413930292765144, train_accuracy=0.75, train_precision=0.8269230769230769, train_recall=0.75, val_loss=0.21982724720146507, val_accuracy=0.25, val_precision=0.0625, val_recall=0.25
1/1 [==============================] - 0s 171ms/step


c:\Users\treym\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


1/1 [==============================] - 0s 176ms/step
Batch 30: train_loss=0.4560761227621697, train_accuracy=0.84375, train_precision=0.8777173913043479, train_recall=0.84375, val_loss=0.20685036340728402, val_accuracy=0.25, val_precision=0.0625, val_recall=0.25
1/1 [==============================] - 0s 182ms/step
Epoch 2: train_loss=0.4701090050439234, train_accuracy=0.875, train_precision=0.8977272727272727, train_recall=0.875, val_loss=0.26686917117331177, val_accuracy=0.375, val_precision=0.8214285714285714, val_recall=0.375
32/32 [==============================] - 139s 4s/step - loss: 0.0189 - accuracy: 1.0000 - val_loss: 0.1587 - val_accuracy: 0.3750
